# Entrega 2 — Maceta Inteligente
Modelos de clasificación supervisada para predecir la condición de la planta.

- Joaquin Bertaux
- Ignacio Borreani
- Franco Rodriguez
- Santiago Silvera

## Preparación de Datos

Pasos:
1. Carga y limpieza del dataset
2. Tratamiento de outliers
3. Codificación de la variable objetivo
4. División train/test
5. Escalado de features

### 1. Carga y limpieza

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

col_names = ['temperatura', 'humedad', 'luz', 'humedad_suelo', 'label']
col_names_ts = ['timestamp'] + col_names

archivos = {
    'datos1.csv': dict(header=None, names=col_names_ts),
    'datos2.csv': dict(header=None, names=col_names_ts),
    'datos3.csv': dict(header=0,    names=col_names),
    'datos4.csv': dict(header=None, names=col_names_ts),
    'datos5.csv': dict(header=None, names=col_names_ts),
}

partes = []
for nombre, kwargs in archivos.items():
    d = pd.read_csv(nombre, **kwargs)
    d['origen'] = nombre
    partes.append(d)

df_raw = pd.concat(partes, ignore_index=True)

# Eliminar columnas no útiles para el modelo
# - humedad: sensor roto, siempre 0.0 (excepto datos3 que tiene lecturas previas al fallo)
# - timestamp: no es una feature, es metadata de la sesión
# - origen: solo indica el archivo fuente
df_clean = df_raw.drop(columns=['humedad', 'timestamp', 'origen'], errors='ignore')

print(f"Shape original: {df_raw.shape}")
print(f"Shape limpio:   {df_clean.shape}")
print(f"\nColumnas finales: {list(df_clean.columns)}")
print(f"\nNulos por columna:\n{df_clean.isnull().sum()}")
print(f"\nDistribución de clases:\n{df_clean['label'].value_counts()}")

Shape original: (924, 7)
Shape limpio:   (924, 4)

Columnas finales: ['temperatura', 'luz', 'humedad_suelo', 'label']

Nulos por columna:
temperatura      0
luz              0
humedad_suelo    0
label            0
dtype: int64

Distribución de clases:
label
Estable    403
Ideal      403
Decaida    118
Name: count, dtype: int64


### 2. Tratamiento de outliers

En el EDA se detectaron 17 outliers en `humedad_suelo` (1.8%). Se usa clipping con los límites IQR en lugar de eliminar filas, para no perder datos de una clase ya minoritaria (Decaida tiene solo 118 registros).

In [3]:
feature_cols = ['temperatura', 'luz', 'humedad_suelo']

df_prep = df_clean.copy()

for col in feature_cols:
    Q1 = df_prep[col].quantile(0.25)
    Q3 = df_prep[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = ((df_prep[col] < lower) | (df_prep[col] > upper)).sum()
    df_prep[col] = df_prep[col].clip(lower, upper)
    print(f"{col}: {before} outliers clippeados → [{lower:.1f}, {upper:.1f}]")

print(f"\nShape final: {df_prep.shape}")

temperatura: 0 outliers clippeados → [-4.6, 41.6]
luz: 0 outliers clippeados → [1296.1, 5229.1]
humedad_suelo: 17 outliers clippeados → [911.9, 3164.9]

Shape final: (924, 4)


### 3. Codificación de la variable objetivo

Se usa encoding **ordinal** que respeta el orden natural de la condición de la planta:

| Label    | Código |
|----------|--------|
| Decaida  | 0      |
| Estable  | 1      |
| Ideal    | 2      |

Esto permite tanto clasificación multiclase (3 clases) como binaria (Decaida vs. no-Decaida) según el modelo.

In [4]:
LABEL_MAP = {'Decaida': 0, 'Estable': 1, 'Ideal': 2}
LABEL_MAP_INV = {v: k for k, v in LABEL_MAP.items()}

df_prep['label_enc'] = df_prep['label'].map(LABEL_MAP)

assert df_prep['label_enc'].isnull().sum() == 0, "Hay labels no reconocidos"

print("Encoding aplicado:")
print(df_prep[['label', 'label_enc']].drop_duplicates().sort_values('label_enc').to_string(index=False))
print(f"\nDistribución:\n{df_prep['label_enc'].value_counts().sort_index().rename(LABEL_MAP_INV)}")

X = df_prep[feature_cols].values
y = df_prep['label_enc'].values
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

Encoding aplicado:
  label  label_enc
Decaida          0
Estable          1
  Ideal          2

Distribución:
label_enc
Decaida    118
Estable    403
Ideal      403
Name: count, dtype: int64

X shape: (924, 3)
y shape: (924,)


### 4. División train/test

Se usa 80/20 con `stratify=y` para preservar la proporción de clases en ambos conjuntos, lo cual es importante dado que *Decaida* es la clase minoritaria.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape[0]} muestras  |  Test: {X_test.shape[0]} muestras")
print("\nDistribución de clases en train:")
for code, name in LABEL_MAP_INV.items():
    n = (y_train == code).sum()
    print(f"  {name} ({code}): {n}  ({n/len(y_train)*100:.1f}%)")
print("\nDistribución de clases en test:")
for code, name in LABEL_MAP_INV.items():
    n = (y_test == code).sum()
    print(f"  {name} ({code}): {n}  ({n/len(y_test)*100:.1f}%)")

Train: 739 muestras  |  Test: 185 muestras

Distribución de clases en train:
  Decaida (0): 95  (12.9%)
  Estable (1): 322  (43.6%)
  Ideal (2): 322  (43.6%)

Distribución de clases en test:
  Decaida (0): 23  (12.4%)
  Estable (1): 81  (43.8%)
  Ideal (2): 81  (43.8%)


### 5. Escalado de features

Se aplica `StandardScaler` **ajustado solo sobre train** para evitar data leakage. El scaler se guarda para aplicarlo también en inferencia (ESP32 necesitará los mismos parámetros: `mean_` y `scale_`).

In [6]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Parámetros del scaler (necesarios para inferencia en ESP32):")
for i, col in enumerate(feature_cols):
    print(f"  {col}: media={scaler.mean_[i]:.4f}, std={scaler.scale_[i]:.4f}")

print(f"\nX_train_sc — media: {X_train_sc.mean(axis=0).round(4)}  (debe ser ~0)")
print(f"X_train_sc — std:   {X_train_sc.std(axis=0).round(4)}   (debe ser ~1)")

Parámetros del scaler (necesarios para inferencia en ESP32):
  temperatura: media=17.1884, std=6.5506
  luz: media=3156.1110, std=670.8628
  humedad_suelo: media=1910.4063, std=529.8711

X_train_sc — media: [-0. -0.  0.]  (debe ser ~0)
X_train_sc — std:   [1. 1. 1.]   (debe ser ~1)


### Resumen del dataset preparado

In [7]:
print("=" * 50)
print("DATASET LISTO PARA ENTRENAMIENTO")
print("=" * 50)
print(f"  Features:        {feature_cols}")
print(f"  Clases:          {LABEL_MAP}")
print(f"  Total muestras:  {len(X)}")
print(f"  Train:           {len(X_train)}  (con escalado: X_train_sc)")
print(f"  Test:            {len(X_test)}   (con escalado: X_test_sc)")
print()
print("Variables disponibles para los modelos:")
print("  X_train, X_test         → sin escalar (para árboles/RF)")
print("  X_train_sc, X_test_sc   → escaladas   (para KNN, SVM, redes)")

DATASET LISTO PARA ENTRENAMIENTO
  Features:        ['temperatura', 'luz', 'humedad_suelo']
  Clases:          {'Decaida': 0, 'Estable': 1, 'Ideal': 2}
  Total muestras:  924
  Train:           739  (con escalado: X_train_sc)
  Test:            185   (con escalado: X_test_sc)

Variables disponibles para los modelos:
  X_train, X_test         → sin escalar (para árboles/RF)
  X_train_sc, X_test_sc   → escaladas   (para KNN, SVM, redes)
